# Advanced Fine-Tuning: QLoRA & DPO

Full Fine-Tuning of a massive LLM (like a 70B parameter model) requires clusters of A100 GPUs and millions of dollars. To democratize AI, researchers invented techniques to fine-tune massive models on a single consumer GPU.

## 1. QLoRA (Quantized Low-Rank Adaptation)

**Theory:**
QLoRA combines two memory-saving techniques:
1. **Quantization:** Standard weights are 32-bit floating-point numbers. Quantization squeezes these down to 4-bit integers. This massively reduces the RAM required to hold the model (e.g., shrinking a 30GB model to 6GB).
2. **LoRA:** Instead of training all the weights, LoRA freezes the original model and injects tiny, "low-rank" adapter matrices. You only train these tiny adapters.

- **Result:** You can fine-tune a massive LLaMA model on a single 24GB GPU without sacrificing much performance.

In [ ]:
# Conceptual Implementation using HuggingFace PEFT and BitsAndBytes
"""
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# 1. Load in 4-bit (Quantization)
bnb_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3-8b", quantization_config=bnb_config)

# 2. Apply LoRA Adapters
lora_config = LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"])
peft_model = get_peft_model(model, lora_config)

# 3. Train only the peft_model on your dataset
"""

## 2. Direct Preference Optimization (DPO)

**Theory:**
Historically, aligning LLMs to human preferences required **RLHF (Reinforcement Learning from Human Feedback)**. RLHF is notoriously unstable and requires training three separate models (a reward model, a policy model, and a reference model).

- **DPO** achieves the exact same result as RLHF but skips the complex reinforcement learning step entirely. It frames preference learning (e.g., "Answer A is better than Answer B") as a simple binary classification problem.
- **Result:** It is significantly easier to code, more stable to train, and uses far less memory.

In [ ]:
# Conceptual Implementation using HuggingFace TRL (Transformer Reinforcement Learning)
"""
from trl import DPOTrainer

dpo_trainer = DPOTrainer(
    model=peft_model,
    ref_model=None, # TRL automatically handles the reference model for LoRA
    train_dataset=preference_dataset, # A dataset containing 'prompt', 'chosen_answer', and 'rejected_answer'
    args=training_args
)
dpo_trainer.train()
"""